# Modelo LSTM: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.lstm_wrapper import LSTMModel
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import test_toolkit as tk
from src.data.dataset import TimeSeriesDataset

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # irrelevante en CPU, pero fija por si acaso
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

device = torch.device("cpu")
torch.set_default_device("cpu")
assert not torch.cuda.is_available(), "Este notebook debe correr en CPU; hay una GPU visible."

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "LSTM"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float32")
rs_train = df_train["rs_daily"].values.astype("float32")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float32")
rs_test = df_test["rs_daily"].values.astype("float32")

In [2]:
assert len(y_train) == len(rs_train), "y_train y rs_train desalineados"
assert len(y_test) == len(rs_test), "y_test y rs_test desalineados"
assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"
assert df_train.index.is_monotonic_increasing, "train no está ordenado cronológicamente"
assert df_test.index.is_monotonic_increasing, "test no está ordenado cronológicamente"

print("=" * 70)
print(f"Device: {device}  |  CUDA disponible: {torch.cuda.is_available()}")
print(f"PyTorch {torch.__version__}  |  NumPy {np.__version__}  |  pandas {pd.__version__}")
print(f"Train: {len(y_train):>6} obs  ({df_train.index[0].date()} → {df_train.index[-1].date()})")
print(f"Test:  {len(y_test):>6} obs  ({df_test.index[0].date()} → {df_test.index[-1].date()})")
print(f"Resultados: {RESULTS_PATH}")
print(f"Modelos:    {MODELS_PATH}")
print(f"Seed fija:  {SEED}")
print("=" * 70)

Device: cpu  |  CUDA disponible: False
PyTorch 2.13.0  |  NumPy 2.2.6  |  pandas 2.3.3
Train:   2171 obs  (2014-01-03 → 2022-11-03)
Test:     953 obs  (2022-11-07 → 2026-08-21)
Resultados: C:\Users\jcami\Documents\brent-crude-nlp-market-dynamics\results\LSTM
Modelos:    C:\Users\jcami\Documents\brent-crude-nlp-market-dynamics\models
Seed fija:  42


In [3]:
print("=" * 70)
print("PRE-FLIGHT: verificando LSTMModel")
print("=" * 70)

m = LSTMModel(
    look_back=7, hidden_size=32, num_layers=1,
    dropout=0.0, learning_rate=1e-3, weight_decay=1e-6,
    max_epochs=20, patience=3, retrain_freq=20, batch_size=64,
)

print(f"  context_length = {m.context_length}")
print(f"  device         = {m.device}")

with torch.no_grad():
    x = torch.randn(4, m.context_length, 1)
    out = m.net(x)

print(f"  input  shape   = {tuple(x.shape)}")
print(f"  output shape   = {tuple(out.shape)}")
print(f"  output min     = {out.min().item():.3e}")
print(f"  output max     = {out.max().item():.3e}")
print(f"  ¿todos > 0?    = {bool((out > 0).all())}")

assert out.shape == (4, 1), f"Shape incorrecto: {out.shape} (esperado (4, 1))"
assert (out > 0).all(), "Hay valores ≤ 0 → softplus no está funcionando"
assert torch.isfinite(out).all(), "Hay NaN/Inf en la salida"

# Test QLIKELoss con esa salida
from src.eval.losses import QLIKELoss
criterion = QLIKELoss(eps=1e-8)
y_true = torch.rand(4, 1) * 1e-4  # RS típicos
loss = criterion(out, y_true)
print(f"  QLIKELoss      = {loss.item():.6f}")
assert torch.isfinite(loss), "QLIKELoss es NaN/Inf"

print("\nPRE-FLIGHT PASÓ — el modelo produce varianzas positivas y la pérdida es finita")

PRE-FLIGHT: verificando LSTMModel
  context_length = 7
  device         = cpu
  input  shape   = (4, 7, 1)
  output shape   = (4, 1)
  output min     = 6.703e-01
  output max     = 6.718e-01
  ¿todos > 0?    = True
  QLIKELoss      = 9.594364

PRE-FLIGHT PASÓ — el modelo produce varianzas positivas y la pérdida es finita


In [3]:
import numpy as np
from src.models.lstm_wrapper import LSTMModel
from src.eval.scorer import VolatilityScorer

y_smoke = y_train[:600]
rs_smoke = rs_train[:600]

params = {
    "look_back": 7, "hidden_size": 32, "num_layers": 1,
    "dropout": 0.0, "learning_rate": 1e-3, "weight_decay": 1e-6,
    "max_epochs": 20, "patience": 3, "retrain_freq": 20, "batch_size": 64,
}

# Fold 1: warmup=300, val_size=150
y_train_fold = y_smoke[0:299]
rs_train_fold = rs_smoke[0:299]
y_val_fold   = y_smoke[300:450]
rs_val_fold  = rs_smoke[300:450]

print(f"Train: {len(y_train_fold)}  Val: {len(y_val_fold)}")
print(f"rs_train: min={rs_train_fold.min():.3e} max={rs_train_fold.max():.3e}")
print(f"rs_val:   min={rs_val_fold.min():.3e} max={rs_val_fold.max():.3e}")

print("\n[1] Corriendo fit...")
model = LSTMModel(**params)
model.fit(y_train_fold, rs_train_fold)
print(f"    Fit OK. Converged: {model.get_diagnostics()}")
print(f"    Train loss últimas 3: {model.history['train_loss'][-3:]}")
print(f"    Val loss últimas 3:   {model.history['val_loss'][-3:]}")

print("\n[2] Corriendo forecast...")
context = model.context_length
y_seq = y_smoke[300-context:450]
rs_seq = rs_smoke[300-context:450]
print(f"    y_seq: {len(y_seq)}, rs_seq: {len(rs_seq)}")

preds = model.forecast(y_seq, rs_seq)
print(f"    Preds shape: {preds.shape}")
print(f"    Preds min:   {preds.min():.6e}")
print(f"    Preds max:   {preds.max():.6e}")
print(f"    ¿NaN en preds? {np.isnan(preds).any()}")
print(f"    ¿Inf en preds? {np.isinf(preds).any()}")
print(f"    rs_val shape:  {rs_val_fold.shape}")
print(f"    ¿NaN en rs_val? {np.isnan(rs_val_fold).any()}")

# --- QLIKE ---
print("\n[3] Calculando QLIKE...")
scorer = VolatilityScorer()
qlike = scorer.qlike(rs_val_fold, preds)
print(f"    QLIKE = {qlike}")
print(f"    ¿NaN? {np.isnan(qlike)}")

# --- VEREDICTO ---
print("\n" + "=" * 60)
if np.isnan(preds).any():
    print("FALLO EN FORECAST: preds tiene NaN")
elif np.isnan(qlike):
    print("FALLO EN SCORER: preds finito pero qlike es NaN")
    # Inspección manual
    qlike_manual = np.mean(rs_val_fold / preds - np.log(rs_val_fold / preds) - 1)
    print(f"    QLIKE manual: {qlike_manual}")
    print(f"    ratio min: {(rs_val_fold / preds).min():.6e}")
    print(f"    ratio max: {(rs_val_fold / preds).max():.6e}")
    print(f"    ¿algún ratio ≤ 0? {bool((rs_val_fold / preds <= 0).any())}")
else:
    print(f"✓ FOLD 1 OK — QLIKE = {qlike:.6f}")
print("=" * 60)

Train: 299  Val: 150
rs_train: min=1.841e-06 max=3.523e-03
rs_val:   min=1.735e-05 max=4.037e-03

[1] Corriendo fit...
    Fit OK. Converged: {'converged': False, 'epochs_run': 20}
    Train loss últimas 3: [4.831761121749878, 4.539312303066254, 4.268163204193115]
    Val loss últimas 3:   [2.7984180450439453, 2.5273990631103516, 2.275108575820923]

[2] Corriendo forecast...
    y_seq: 157, rs_seq: 157
    Preds shape: (150,)
    Preds min:   3.997230e-04
    Preds max:   1.455657e-02
    ¿NaN en preds? False
    ¿Inf en preds? False
    rs_val shape:  (150,)
    ¿NaN en rs_val? False

[3] Calculando QLIKE...
    QLIKE = 0.4170290249764145
    ¿NaN? False

✓ FOLD 1 OK — QLIKE = 0.417029


In [4]:
import time
import pandas as pd
import numpy as np
from src.models.lstm_wrapper import LSTMModel
from src.trainer.walk_forward import WalkForwardTrainer
from src.eval.scorer import VolatilityScorer

# ------------------------------------------------------------------ #
# 1. Override de suggest_params para forzar HPs rápidos              #
# ------------------------------------------------------------------ #
_original_suggest_params = LSTMModel.suggest_params

def _fast_suggest_params(trial):
    return {
        "look_back": 7,
        "hidden_size": 32,
        "num_layers": 1,
        "dropout": 0.0,
        "learning_rate": 1e-3,
        "weight_decay": 1e-6,
        "max_epochs": 20,
        "patience": 3,
        "retrain_freq": 20,
        "batch_size": 64,
    }

LSTMModel.suggest_params = staticmethod(_fast_suggest_params)

# ------------------------------------------------------------------ #
# 2. Setup limpio                                                    #
# ------------------------------------------------------------------ #
SMOKE_DIR = RESULTS_PATH / "smoke_test"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
csv_file = SMOKE_DIR / "LSTMModel_results.csv"
if csv_file.exists():
    csv_file.unlink()
    print(f"CSV previo eliminado: {csv_file}")

# ------------------------------------------------------------------ #
# 3. Correr                                                          #
# ------------------------------------------------------------------ #
trainer = WalkForwardTrainer(
    model_class=LSTMModel,
    scorer=VolatilityScorer(),
    output_dir=str(SMOKE_DIR),
    n_splits=2,
    warmup=300,
    embargo=1,
    n_trials=1,
)

y_smoke = y_train[:600]
rs_smoke = rs_train[:600]

print(f"\nSmoke data: {len(y_smoke)} samples  "
      f"(warmup=300, n_splits=2, n_trials=1)")
print(f"Esperando ~150 predicciones por fold, ~2 min en CPU...\n")

t0 = time.time()
trainer.run(y_smoke, rs_smoke)
elapsed = time.time() - t0

# ------------------------------------------------------------------ #
# 4. Restaurar suggest_params original                               #
# ------------------------------------------------------------------ #
LSTMModel.suggest_params = _original_suggest_params

# ------------------------------------------------------------------ #
# 5. Verificar                                                       #
# ------------------------------------------------------------------ #
print("\n" + "=" * 70)
print(f"RESULTADOS DEL SMOKE TEST (tiempo total: {elapsed:.1f}s)")
print("=" * 70)

df = pd.read_csv(csv_file)
print(df.to_string(index=False))
print("=" * 70)

# Verificaciones (converged se omite porque max_epochs=20 no da tiempo a early stopping)
assert not df["qlike"].isna().any(), "QLIKE tiene NaN"
assert (df["qlike"] > 0).all(), "QLIKE tiene valores ≤ 0"
assert len(df) == 2, f"Esperadas 2 filas, hay {len(df)}"

print(f"\n✓ QLIKE por fold:  {[round(x, 6) for x in df['qlike'].tolist()]}")
print(f"✓ QLIKE medio:     {df['qlike'].mean():.6f}")
print(f"✓ Épocas:          {df['epochs_run'].tolist()}")
print(f"✓ Train time:      {df['train_time_sec'].tolist()} s")
print(f"✓ Tiempo total:    {elapsed:.1f} s")

if df["qlike"].mean() < 9.0:
    print("\n SMOKE TEST PASÓ — los scripts están en sintonía y el modelo aprende.")
else:
    print("\n SMOKE TEST CORRIÓ pero QLIKE ≈ 9.6 (revisar hiperparámetros o datos).")

CSV previo eliminado: C:\Users\jcami\Documents\brent-crude-nlp-market-dynamics\results\LSTM\smoke_test\LSTMModel_results.csv

Smoke data: 600 samples  (warmup=300, n_splits=2, n_trials=1)
Esperando ~150 predicciones por fold, ~2 min en CPU...


RESULTADOS DEL SMOKE TEST (tiempo total: 34.4s)
 look_back  hidden_size  num_layers  dropout  learning_rate  weight_decay  max_epochs  patience  retrain_freq  batch_size  fold  train_size  val_size  converged  epochs_run  train_time_sec    qlike
         7           32           1      0.0          0.001      0.000001          20         3            20          64     1         299       150      False          20            2.11 0.410289
         7           32           1      0.0          0.001      0.000001          20         3            20          64     2         449       150      False          20            1.78 0.913619

✓ QLIKE por fold:  [0.410289, 0.913619]
✓ QLIKE medio:     0.661954
✓ Épocas:          [20, 20]
✓ Train time:   

In [5]:
"""
build_onnx
==========

Exports the trained PyTorch model to ONNX and runs a strict round-trip test.
"""

import json
import numpy as np
import torch
import onnxruntime as ort
from pathlib import Path
from src.models.lstm_wrapper import LSTMNetwork

def export_and_verify(artifact_dir: str | Path):
    artifact_path = Path(artifact_dir)
    meta_path = artifact_path / "metadata.json"
    weights_path = artifact_path / "model_state_dict.pt"
    
    if not meta_path.exists() or not weights_path.exists():
        raise FileNotFoundError(f"Faltan artefactos en {artifact_path}")

    # 1. Cargar metadata
    with open(meta_path, "r") as f:
        metadata = json.load(f)

    # 2. Reconstruir el modelo
    params = metadata["params"]
    net = LSTMNetwork(
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"]
    )
    net.load_state_dict(torch.load(weights_path, map_location="cpu"))
    net.eval()

    # 3. Exportar a ONNX
    context_len = metadata["context_length"]
    dummy_input = torch.randn(1, context_len, 1, dtype=torch.float32)
    onnx_path = artifact_path / "model.onnx"
    
    print(f"Exportando a {onnx_path}...")
    torch.onnx.export(
        net,
        dummy_input,
        str(onnx_path),
        input_names=["input"],
        output_names=["variance"],
        dynamic_axes={"input": {0: "batch_size"}, "variance": {0: "batch_size"}},
        opset_version=17,
        do_constant_folding=True,
    )

    # 4. Round-Trip Test
    print("Ejecutando Round-Trip Test (PyTorch vs ONNX)...")
    
    # Generar 100 muestras aleatorias de test
    test_inputs = torch.randn(100, context_len, 1, dtype=torch.float32)
    
    # PyTorch
    with torch.no_grad():
        torch_out = net(test_inputs).numpy()
        
    # ONNX Runtime
    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    onnx_out = session.run(None, {"input": test_inputs.numpy()})[0]
    
    # Comparar
    max_diff = np.max(np.abs(torch_out - onnx_out))
    tolerance = 1e-4
    
    print(f"  - Diferencia máxima absoluta: {max_diff:.2e}")
    print(f"  - Tolerancia exigida:         {tolerance:.2e}")
    
    if max_diff > tolerance:
        print("  ❌ ERROR: El modelo ONNX difiere del modelo PyTorch.")
        raise RuntimeError("Round-trip test falló.")
    else:
        print("  ✅ Round-trip test PASÓ. El ONNX es idéntico.")

    # 5. Actualizar metadata.json
    metadata["roundtrip_test"] = {
        "max_abs_error": float(max_diff),
        "tolerance": float(tolerance),
        "passed": True
    }
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
        
    print(f"✓ Metadata actualizada con resultados del test.")
    return onnx_path

# Ejecutar
export_and_verify(MODELS_PATH / "lstm_final")

ModuleNotFoundError: No module named 'onnxruntime'